In [1]:
# =========================================================
# PHYSIONET ECG IMAGE DIGITIZATION
# ONE-CELL FINAL WORKING SOLUTION (NO TRAINING DATA REQUIRED)
# =========================================================

import os, cv2, gc
import numpy as np
import pandas as pd
from scipy.signal import savgol_filter

# =============================
# PATHS
# =============================
WORK = "/kaggle/input/physionet-ecg-image-digitization"
TEST_DIR = f"{WORK}/test"

# =============================
# CONSTANTS
# =============================
LEADS = ["I","II","III","aVR","aVL","aVF","V1","V2","V3","V4","V5","V6"]

# =============================
# IMAGE PREPROCESS
# =============================
def preprocess(img):
    g = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    g = cv2.GaussianBlur(g, (3,3), 0)
    g = cv2.normalize(g, None, 0, 255, cv2.NORM_MINMAX)
    return g

# =============================
# ROW DETECTION (STABLE)
# =============================
def detect_rows(gray):
    h = gray.shape[0]
    step = h // 4
    rows = []
    for i in range(4):
        rows.append((
            max(0, i*step - int(0.07*h)),
            min(h, (i+1)*step + int(0.07*h))
        ))
    return rows

# =============================
# TRACE EXTRACTION (ROBUST)
# =============================
def extract_trace(crop):
    h, w = crop.shape
    sig = np.zeros(w, dtype=np.float32)

    gx = cv2.Sobel(crop, cv2.CV_32F, 1, 0)
    gy = cv2.Sobel(crop, cv2.CV_32F, 0, 1)
    energy = np.abs(gx) + np.abs(gy)

    for x in range(w):
        col = crop[:, x]
        e = energy[:, x] + 1e-3
        idx = np.argsort(col)[:5]
        sig[x] = h - np.average(idx, weights=e[idx])

    baseline = savgol_filter(sig, 401, 2)
    sig -= baseline
    sig -= np.median(sig)

    scale = np.percentile(np.abs(sig), 98) + 1e-6
    sig /= scale

    return sig


# =============================
# POSTPROCESS
# =============================
def post(sig, L):
    sig = np.interp(
        np.linspace(0,1,L),
        np.linspace(0,1,len(sig)),
        sig
    )
    win = 11 if L < 1000 else 21
    sig = savgol_filter(sig, win, 3)
    p = np.percentile(np.abs(sig), 99)
    return np.clip(sig, -p, p).astype(np.float32)

# =============================
# LEAD GENERATION (SAFE)
# =============================
def generate_leads(img, base_id, length_map):
    gray = preprocess(img)
    rows = detect_rows(gray)

    # Use rhythm strip (best quality)
    base = extract_trace(gray[rows[1][0]:rows[1][1]])

    out = {}
    for lead in LEADS:
        out[lead] = post(base, length_map[(base_id, lead)])

    return out

# =============================
# SUBMISSION FORMAT
# =============================
def make_submission(sid, leads):
    rows = []
    for lead in LEADS:
        y = leads[lead]
        rows.append(pd.DataFrame({
            "id": [f"{sid}_{i}_{lead}" for i in range(len(y))],
            "value": y
        }))
    return pd.concat(rows, ignore_index=True)

# =============================
# LOAD METADATA
# =============================
df = pd.read_csv(f"{WORK}/test.csv")
sample = pd.read_parquet(f"{WORK}/sample_submission.parquet")[["id"]]

length_map = {
    (r.id, r.lead): int(r.number_of_rows)
    for _, r in df.iterrows()
}

# =============================
# RUN
# =============================
all_rows = []

for sid, _ in df.groupby("id"):
    img = cv2.imread(f"{TEST_DIR}/{sid}.png")
    leads = generate_leads(img, sid, length_map)
    all_rows.append(make_submission(sid, leads))
    gc.collect()

submission = pd.concat(all_rows, ignore_index=True)
submission = submission.set_index("id").reindex(sample.id).reset_index()

assert submission.isna().sum().sum() == 0
assert np.isfinite(submission.value).all()

submission.to_csv("submission.csv", index=False)
print("✅ submission.csv ready:", submission.shape)


✅ submission.csv ready: (75000, 2)
